# Chapter 2: An Array of Sequences
## Fluent Python, 2nd Edition -- Distilled Interactive Notebook

> "Texts, lists and tables together are called 'trains'."
> -- *ABC Programmer's Handbook*

Related wiki articles: [[list-comprehensions-genexps]], [[tuples-as-records-and-immutable-lists]], [[sequence-unpacking]], [[pattern-matching-sequences]], [[slicing]], [[augmented-assignment-sequences]], [[sorting-list-sort-vs-sorted]], [[arrays-deques-alternative-sequences]]

## TL;DR

- **List comprehensions** (listcomps) and **generator expressions** (genexps) are the idiomatic way to build and initialize sequences.
- **Tuples** serve double duty: as **records** (position = meaning) and as **immutable lists** (compact, hashable when contents are immutable).
- **Unpacking** (parallel assignment, `*rest`, nested) eliminates error-prone indexing.
- **Pattern matching** (`match/case`, Python 3.10+) adds declarative destructuring with type checks and guards.
- **Slicing** is more powerful than most realize: named slices, stride, and slice assignment.
- **`+=` and `*=`** behave differently on mutable vs. immutable sequences (`__iadd__` vs. `__add__`).
- **`list.sort()`** mutates in place (returns `None`); **`sorted()`** returns a new list from any iterable.
- Use **`array.array`** for large homogeneous numeric data, **`collections.deque`** for fast O(1) appends/pops at both ends.

---
## 1. List Comprehensions and Generator Expressions

A **listcomp** is a concise, readable way to build a list by filtering and transforming items from an iterable. A **genexp** uses the same syntax in parentheses, yielding items lazily.

See also: [[list-comprehensions-genexps]]

In [ ]:
# Basic listcomp: build a list of Unicode code points
symbols = '$¢£¥€¤'
codes = [ord(s) for s in symbols]
print("Symbols:", symbols)
print("Code points:", codes)

In [ ]:
# Listcomp with filtering (replaces map + filter)
symbols = '$¢£¥€¤'
beyond_ascii = [ord(s) for s in symbols if ord(s) > 127]
print("Beyond ASCII:", beyond_ascii)

# Equivalent using map/filter -- less readable
beyond_ascii_mf = list(filter(lambda c: c > 127, map(ord, symbols)))
print("map/filter:", beyond_ascii_mf)
assert beyond_ascii == beyond_ascii_mf

In [ ]:
# Cartesian product with a listcomp
colors = ['black', 'white']
sizes = ['S', 'M', 'L']
tshirts = [(color, size) for color in colors for size in sizes]
print("T-shirts (color first):", tshirts)

# Reorder the for clauses to sort by size first
tshirts_by_size = [(color, size) for size in sizes for color in colors]
print("T-shirts (size first):", tshirts_by_size)

In [ ]:
# Generator expressions -- lazy, memory-efficient
# genexp to build a tuple
symbols = '$¢£¥€¤'
print(tuple(ord(s) for s in symbols))

# genexp to build an array
import array
print(array.array('I', (ord(s) for s in symbols)))

# genexp in a for loop -- never builds the full list in memory
colors = ['black', 'white']
sizes = ['S', 'M', 'L']
for tshirt in (f'{c} {s}' for c in colors for s in sizes):
    print(tshirt)

**Key insight:** Listcomps have their own local scope in Python 3. The loop variable does not leak. But the walrus operator `:=` *does* persist outside the comprehension.

In [ ]:
# Scope demo: loop var does not leak
x = 'ABC'
codes = [ord(x) for x in x]
print("x is still:", x)  # x not clobbered

# But walrus operator persists
codes = [last := ord(c) for c in x]
print("last =", last)  # 67 -- the ord of 'C'
# print(c)  # would raise NameError -- c only existed inside the listcomp

---
## 2. Tuples as Records and Immutable Lists

Tuples serve **two roles**:
1. **Records** -- position gives meaning (e.g., `(latitude, longitude)`).
2. **Immutable lists** -- more compact, faster, and hashable (when all items are immutable).

See also: [[tuples-as-records-and-immutable-lists]]

In [ ]:
# Tuple as record: each position has meaning
lax_coordinates = (33.9425, -118.408056)
city, year, pop, chg, area = ('Tokyo', 2003, 32_450, 0.66, 8014)
print(f"{city}: population {pop}k in {year}, area {area} km²")

# Tuple unpacking in a for loop
traveler_ids = [('USA', '31195855'), ('BRA', 'CE342567'), ('ESP', 'XDA205856')]
for country, _ in traveler_ids:
    print(country)

In [ ]:
# Tuple as immutable list -- performance and clarity
# tuple(t) returns same object; list(l) always copies
t = (1, 2, 3)
print("Same object?", tuple(t) is t)  # True -- no copy!

l = [1, 2, 3]
print("Same object?", list(l) is l)   # False -- new copy

# Tuples use less memory
import sys
print(f"tuple of 100 ints: {sys.getsizeof(tuple(range(100)))} bytes")
print(f"list  of 100 ints: {sys.getsizeof(list(range(100)))} bytes")

In [ ]:
# CAUTION: tuple immutability only applies to references, not to mutable contents
a = (10, 'alpha', [1, 2])
b = (10, 'alpha', [1, 2])
print("a == b:", a == b)   # True

b[-1].append(99)
print("a == b:", a == b)   # False -- the list inside b changed!
print("b =", b)

# Check if a tuple has a fixed value with hash()
def fixed(o):
    try:
        hash(o)
    except TypeError:
        return False
    return True

print("(10, 'alpha', (1, 2)) fixed?", fixed((10, 'alpha', (1, 2))))
print("(10, 'alpha', [1, 2]) fixed?", fixed((10, 'alpha', [1, 2])))

---
## 3. Sequence and Iterable Unpacking

Unpacking avoids error-prone indexing. Works with **any** iterable.

See also: [[sequence-unpacking]]

In [ ]:
# Parallel assignment
latitude, longitude = (33.9425, -118.408056)
print(f"LAX: {latitude}, {longitude}")

# Swap without temp variable
a, b = 1, 2
b, a = a, b
print(f"After swap: a={a}, b={b}")

# Unpack into function call with *
t = (20, 8)
quotient, remainder = divmod(*t)
print(f"divmod(20, 8) = {quotient} remainder {remainder}")

In [ ]:
# * to grab excess items
a, b, *rest = range(5)
print(f"a={a}, b={b}, rest={rest}")

a, *body, c, d = range(5)
print(f"a={a}, body={body}, c={c}, d={d}")

*head, b, c, d = range(5)
print(f"head={head}, b={b}, c={c}, d={d}")

In [ ]:
# Nested unpacking
metro_areas = [
    ('Tokyo', 'JP', 36.933, (35.689722, 139.691667)),
    ('Delhi NCR', 'IN', 21.935, (28.613889, 77.208889)),
    ('Mexico City', 'MX', 20.142, (19.433333, -99.133333)),
    ('New York-Newark', 'US', 20.104, (40.808611, -74.020386)),
]

print(f"{''!s:15} | {'latitude':>9} | {'longitude':>9}")
for name, _, _, (lat, lon) in metro_areas:
    if lon <= 0:
        print(f"{name:15} | {lat:9.4f} | {lon:9.4f}")

In [ ]:
# * in function calls and sequence literals (PEP 448)
def fun(a, b, c, d, *rest):
    return a, b, c, d, rest

print(fun(*[1, 2], 3, *range(4, 7)))

# * in literals
print(*range(4), 4)         # prints: 0 1 2 3 4
print([*range(4), 4])       # list: [0, 1, 2, 3, 4]
print({*range(4), 4, *(5, 6, 7)})  # set

---
## 4. Pattern Matching with Sequences (Python 3.10+)

`match/case` provides **declarative destructuring**: the shape of the pattern mirrors the shape of the data. Supports guards (`if`), type checks, wildcards (`_`), and `*` for variable-length matching.

See also: [[pattern-matching-sequences]]

In [ ]:
# Pattern matching with sequences -- robot command example
def handle_command(message):
    match message:
        case ['BEEPER', frequency, times]:
            return f"Beep at {frequency}Hz, {times} times"
        case ['NECK', angle]:
            return f"Rotate neck to {angle} degrees"
        case ['LED', ident, intensity]:
            return f"Set LED {ident} brightness to {intensity}"
        case ['LED', ident, red, green, blue]:
            return f"Set LED {ident} color to ({red}, {green}, {blue})"
        case _:
            return f"Unknown command: {message}"

# Test it
commands = [
    ['BEEPER', 440, 3],
    ['NECK', 90],
    ['LED', 2, 0.5],
    ['LED', 1, 255, 128, 0],
    ['DANCE'],
]
for cmd in commands:
    print(f"{str(cmd):40} -> {handle_command(cmd)}")

In [ ]:
# Pattern matching with guard and nested destructuring
metro_areas = [
    ('Tokyo', 'JP', 36.933, (35.689722, 139.691667)),
    ('Mexico City', 'MX', 20.142, (19.433333, -99.133333)),
    ('New York-Newark', 'US', 20.104, (40.808611, -74.020386)),
    ('São Paulo', 'BR', 19.649, (-23.547778, -46.635833)),
]

print(f"{''!s:15} | {'latitude':>9} | {'longitude':>9}")
for record in metro_areas:
    match record:
        case [name, _, _, (lat, lon)] if lon <= 0:
            print(f"{name:15} | {lat:9.4f} | {lon:9.4f}")

---
## 5. Slicing

Slicing is one of Python's most powerful features. Beyond the basic `s[a:b]`, there are strides, named slice objects, and slice assignment.

See also: [[slicing]]

In [ ]:
# Basic slicing with stride
s = 'bicycle'
print(s[::3])   # 'bye'  -- every 3rd character
print(s[::-1])  # 'elcycib'  -- reversed
print(s[::-2])  # 'eccb'  -- reversed, every 2nd

# Split a sequence cleanly at any index x: my_list[:x] + my_list[x:]
nums = [10, 20, 30, 40, 50, 60]
x = 3
print(f"Split at {x}: {nums[:x]} + {nums[x:]}")

In [ ]:
# Named slice objects for readability (like naming spreadsheet ranges)
invoice = """
0.....6.................................40........52...55........
1909  Pimoroni PiBrella                     $17.50    3    $52.50
1489  6mm Tactile Switch x20                 $4.95    2     $9.90
1510  Panavise Jr. - PV-201                 $28.00    1    $28.00
1601  PiTFT Mini Kit 320x240                $34.95    1    $34.95
"""

SKU         = slice(0, 6)
DESCRIPTION = slice(6, 40)
UNIT_PRICE  = slice(40, 52)
QUANTITY    = slice(52, 55)
ITEM_TOTAL  = slice(55, None)

line_items = invoice.split('\n')[2:]
for item in line_items:
    if item.strip():
        print(item[UNIT_PRICE], item[DESCRIPTION])

In [ ]:
# Assigning to slices (mutable sequences only)
l = list(range(10))
print("Original:", l)

l[2:5] = [20, 30]
print("l[2:5] = [20, 30]:", l)

del l[5:7]
print("del l[5:7]:", l)

l[3::2] = [11, 22]
print("l[3::2] = [11, 22]:", l)

# Must assign an iterable (even for one item)
l[2:5] = [100]
print("l[2:5] = [100]:", l)

---
## 6. Augmented Assignment with Sequences

`+=` calls `__iadd__` (in-place) on mutable sequences but `__add__` (creates new object) on immutable ones. This leads to the famous **tuple += puzzler**.

See also: [[augmented-assignment-sequences]]

In [ ]:
# Mutable: += modifies in place (same object id)
l = [1, 2, 3]
original_id = id(l)
l *= 2
print(f"List: {l}, same object? {id(l) == original_id}")

# Immutable: *= creates a new object
t = (1, 2, 3)
original_id = id(t)
t *= 2
print(f"Tuple: {t}, same object? {id(t) == original_id}")

In [ ]:
# The += puzzler: both A and B happen!
t = (1, 2, [30, 40])
try:
    t[2] += [50, 60]
except TypeError as e:
    print(f"TypeError: {e}")

# But the list WAS modified before the error!
print(f"t = {t}")
# Lesson: avoid putting mutable items in tuples

**What happened?** Python's bytecode first does `t[2].extend([50, 60])` (succeeds because the list is mutable), then tries `t[2] = result` (fails because the tuple is immutable). The operation is **not atomic**.

---
## 7. Sorting: list.sort vs sorted

- `list.sort()` -- in-place, returns `None` (API convention for mutation).
- `sorted(iterable)` -- returns a new list, works on any iterable.
- Both accept `key` and `reverse` keyword arguments.
- Python uses **Timsort** -- a stable, adaptive sort.

See also: [[sorting-list-sort-vs-sorted]]

In [ ]:
fruits = ['grape', 'raspberry', 'apple', 'banana']

# sorted() returns a new list -- original unchanged
print("sorted:", sorted(fruits))
print("original:", fruits)

# Sorting by length (stable: equal-length items keep original order)
print("by length:", sorted(fruits, key=len))

# Reverse sort
print("reverse:", sorted(fruits, reverse=True))

# list.sort() mutates in place, returns None
result = fruits.sort()
print("After .sort():", fruits)
print("Return value:", result)  # None!

In [ ]:
# key function examples
words = ['Banana', 'apple', 'Cherry', 'date']

# Case-insensitive sort
print("case-insensitive:", sorted(words, key=str.lower))

# Sort by last character
print("by last char:", sorted(words, key=lambda w: w[-1]))

# Stability demo: items that compare equal keep their relative order
data = [('b', 2), ('a', 3), ('c', 1), ('b', 1), ('a', 2)]
print("sort by name:", sorted(data, key=lambda x: x[0]))

---
## 8. Arrays, Deques, and Alternative Sequence Types

Choose the right container:
- **`array.array`** -- compact storage for millions of numbers (no per-object overhead).
- **`collections.deque`** -- O(1) append/pop at both ends, thread-safe, optionally bounded.
- **`queue.Queue`**, **`heapq`** -- for concurrency and priority queues.

See also: [[arrays-deques-alternative-sequences]]

In [ ]:
# array.array -- compact numeric storage
from array import array
from random import random

# Create array of doubles from a genexp
floats = array('d', (random() for _ in range(10)))
print("Last float:", floats[-1])
print("Type code:", floats.typecode)

# Compare memory usage
import sys
n = 1000
as_list = [float(i) for i in range(n)]
as_array = array('d', range(n))
print(f"\nlist of {n} floats:  {sys.getsizeof(as_list):>6} bytes")
print(f"array of {n} floats: {sys.getsizeof(as_array):>6} bytes")

In [ ]:
# collections.deque -- double-ended queue
from collections import deque

# Bounded deque -- discards from opposite end when full
dq = deque(range(10), maxlen=10)
print("Initial:", dq)

dq.rotate(3)  # move 3 from right to left
print("rotate(3):", dq)

dq.rotate(-4)  # move 4 from left to right
print("rotate(-4):", dq)

dq.appendleft(-1)  # full deque: rightmost item dropped
print("appendleft(-1):", dq)

dq.extend([11, 22, 33])  # 3 added right, 3 dropped left
print("extend([11,22,33]):", dq)

dq.extendleft([10, 20, 30, 40])  # items reversed on left
print("extendleft([10,20,30,40]):", dq)

In [ ]:
# memoryview -- shared memory, no copying
from array import array

# Create alternate views on the same memory
octets = array('B', range(6))
m1 = memoryview(octets)
m2 = m1.cast('B', [2, 3])
m3 = m1.cast('B', [3, 2])

print("m2 (2x3):", m2.tolist())
print("m3 (3x2):", m3.tolist())

# Modify through one view -- all views see the change
m2[1, 1] = 22
m3[1, 1] = 33
print("octets after changes through views:", octets)

---
## Exercises

Test your understanding of Chapter 2 concepts.

### Exercise 1: Listcomp Challenge
Use a single list comprehension to produce all Pythagorean triples `(a, b, c)` where `a`, `b`, `c` are all <= 20 and `a <= b`.

In [ ]:
# Exercise 1: Your solution here
# triples = [...]

# Solution:
triples = [(a, b, c)
           for a in range(1, 21)
           for b in range(a, 21)
           for c in range(b, 21)
           if a**2 + b**2 == c**2]
print(f"Found {len(triples)} triples:")
for t in triples:
    print(t)

### Exercise 2: Unpacking Practice
Given the data below, use nested unpacking to print each student's name and their highest grade.

In [ ]:
# Exercise 2
students = [
    ('Alice', 'CS', (92, 88, 95)),
    ('Bob', 'Math', (78, 85, 82)),
    ('Carol', 'CS', (90, 91, 89)),
]

# Solution:
for name, dept, grades in students:
    print(f"{name} ({dept}): highest grade = {max(grades)}")

### Exercise 3: Slice Assignment
Start with `list(range(10))`. Using only slice assignment (no loops), replace elements at indices 1, 3, 5, 7 with `'a'`, `'b'`, `'c'`, `'d'`.

In [ ]:
# Exercise 3
nums = list(range(10))
print("Before:", nums)

# Solution: use stride-2 slice starting at index 1
nums[1::2] = ['a', 'b', 'c', 'd']
print("After: ", nums)

### Exercise 4: Sorting Challenge
Sort a list of `(name, age)` tuples by age (ascending), breaking ties by name (alphabetically).

In [ ]:
# Exercise 4
people = [('Charlie', 30), ('Alice', 25), ('Bob', 30), ('Diana', 25)]

# Solution: tuple comparison is lexicographic, so sort by (age, name)
result = sorted(people, key=lambda p: (p[1], p[0]))
print(result)

---
## Key Takeaways

1. **Prefer listcomps and genexps** over `map`/`filter` for readability and performance.
2. **Tuples are not just immutable lists** -- use them as records where position carries meaning.
3. **Unpacking eliminates indexing** -- use parallel assignment, `*rest`, and nested unpacking.
4. **Pattern matching** (`match/case`) is declarative destructuring -- the pattern mirrors the data shape.
5. **Named slices** (`SKU = slice(0, 6)`) make flat-file parsing readable.
6. **`+=` is not atomic** on tuples containing mutable items -- avoid mutable items in tuples.
7. **`list.sort()` vs `sorted()`**: in-place (returns `None`) vs new list (any iterable). Both are stable.
8. **Choose the right sequence**: `array.array` for numbers, `deque` for queue patterns, `list` for general use.

See also: [[list-comprehensions-genexps]], [[tuples-as-records-and-immutable-lists]], [[sequence-unpacking]], [[pattern-matching-sequences]], [[slicing]], [[augmented-assignment-sequences]], [[sorting-list-sort-vs-sorted]], [[arrays-deques-alternative-sequences]]